# §13.2.4 — 스케일 제거 시 어텐션 엔트로피 붕괴의 재현

> 딥러닝 교재 · 3부 13장 2절 4항 (🐍)
> 선행: §13.2.1(내적 분산 = $d$) · §13.2.2(포화와 야코비안 붕괴) · §13.2.5(가정의 유효 기간)

## 이 노트북이 답하는 질문

1. **스케일이 없으면 초기 어텐션 엔트로피는 정말 $d$와 함께 붕괴하는가?**
2. **포화한 소프트맥스를 지난 기울기는 얼마나 죽는가?**
3. **그 결과 학습은 실제로 느려지거나 실패하는가?**

**예상 실행 시간** CPU 약 70초 (`FAST = True`이면 약 40초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제와 모델 — 어텐션이 답을 찾아야 하는 분류

길이 $T$의 열에서 한 위치에 표지(marker) 채널이 켜져 있고, 정답은 **표지된 위치의
내용 부류**다. 모델은 학습 가능한 질의 $u$ 하나로 열을 조회하는 어텐션 풀링 분류기:
$\alpha=\operatorname{softmax}\!\big(u^\top k_j/\text{scale}\big)$, $\hat y=W\!\sum_j\alpha_j v_j$.
표지를 찾아 $\alpha$를 그 위치에 몰아야만 풀리는 과제이므로, **어디를 볼지의 학습**이
병목이다 — §13.2.2가 죽는다고 말한 바로 그 학습이다.

내용·표지 임베딩은 성분 분산 1의 무작위 고정 벡터로 두어(학습하지 않음), 초기화 시점
가정(§13.2.5)이 정확히 성립하게 한다. 한 시행의 내용들은 서로 겹치지 않게 뽑는다.

In [ ]:
T_SEQ = 16
N_CLS = 16          # 위치마다 서로 다른 내용이 오도록 부류 수 = T

def make_task(d, seed):
    rn = np.random.default_rng(seed)
    E_cls = rn.standard_normal((N_CLS, d))      # 내용 임베딩 (고정)
    e_mark = rn.standard_normal(d)              # 표지 채널 (고정, 더해짐)
    return E_cls, e_mark

def make_batch(E_cls, e_mark, B, rn):
    # 한 시행 안에서 내용이 겹치지 않게 뽑는다 — 겹치면 최대 로짓이 중복되어
    # 포화가 절반에서 멈추는 인공물이 생긴다 (직접 바꿔 확인해 보라)
    cls = np.array([rn.permutation(N_CLS)[:T_SEQ] for _ in range(B)])
    X = E_cls[cls]
    pos = rn.integers(0, T_SEQ, B)
    X[np.arange(B), pos] += e_mark
    y = cls[np.arange(B), pos]
    return X, y

def init_model(d, seed):
    rn = np.random.default_rng(seed + 1)
    return {'Wk': rn.standard_normal((d, d)) / np.sqrt(d),
            'Wv': rn.standard_normal((d, d)) / np.sqrt(d),
            'u':  rn.standard_normal(d),
            'W':  rn.standard_normal((d, N_CLS)) / np.sqrt(d)}

def forward_backward(m, X, y, scale):
    B, T_, d = X.shape
    K = X @ m['Wk']; V = X @ m['Wv']
    e = (K @ m['u']) / scale                    # (B,T)
    e -= e.max(1, keepdims=True)
    a = np.exp(e); a /= a.sum(1, keepdims=True)
    pooled = np.einsum('bt,btd->bd', a, V)
    logits = pooled @ m['W']
    logits -= logits.max(1, keepdims=True)
    P = np.exp(logits); P /= P.sum(1, keepdims=True)
    loss = -np.mean(np.log(P[np.arange(B), y] + 1e-12))
    acc = np.mean(P.argmax(1) == y)
    ent = -(a * np.log(a + 1e-12)).sum(1).mean()     # 어텐션 엔트로피
    # 역전파
    dlog = P.copy(); dlog[np.arange(B), y] -= 1; dlog /= B
    gW = pooled.T @ dlog
    dpool = dlog @ m['W'].T
    da = np.einsum('bd,btd->bt', dpool, V)
    dV = a[:, :, None] * dpool[:, None, :]
    de = a * (da - (a * da).sum(1, keepdims=True)) / scale
    gu = np.einsum('bt,btd->d', de, K)
    dK = de[:, :, None] * m['u'][None, None, :]
    gWk = np.einsum('btd,bte->de', X, dK)
    gWv = np.einsum('btd,bte->de', X, dV)
    grads = {'Wk': gWk, 'Wv': gWv, 'u': gu, 'W': gW}
    jac_mass = (a * (1 - a)).sum(1).mean()     # 야코비안 대각합 = 학습 가능 질량 (식 13.2.2)
    return loss, acc, ent, grads, jac_mass

def train(d, scaled, seed=0, steps=None, lr=2e-3):
    steps = steps or (200 if FAST else 400)
    scale = np.sqrt(d) if scaled else 1.0
    E_cls, e_mark = make_task(d, seed)
    m = init_model(d, seed)
    ms = {k: np.zeros_like(v) for k, v in m.items()}
    vs = {k: np.zeros_like(v) for k, v in m.items()}
    rb = np.random.default_rng(3000 + seed)
    curve_a, curve_e = [], []
    for t in range(1, steps + 1):
        X, y = make_batch(E_cls, e_mark, 64, rb)
        loss, acc, ent, g, _ = forward_backward(m, X, y, scale)
        for k in m:
            ms[k] = 0.9 * ms[k] + 0.1 * g[k]
            vs[k] = 0.999 * vs[k] + 0.001 * g[k] ** 2
            m[k] -= lr * (ms[k] / (1 - 0.9 ** t)) / (np.sqrt(vs[k] / (1 - 0.999 ** t)) + 1e-8)
        if t % 10 == 0 or t == 1:
            curve_a.append((t, acc)); curve_e.append((t, ent))
    return np.array(curve_a), np.array(curve_e)

print(f"과제: T={T_SEQ}, 부류 {N_CLS}, 우연 수준 {1/N_CLS:.3f}")

---
## 2. 초기화 시점 — 엔트로피와 기울기, $d$를 훑으며

학습 전(스텝 0)의 어텐션 엔트로피와, 소프트맥스 야코비안의 학습 가능 질량
$\operatorname{tr}(\operatorname{diag}(\alpha)-\alpha\alpha^\top)=\sum_j\alpha_j(1-\alpha_j)$를
차원별·스케일 유무별로 잰다. 이 대각합이 0이면 로짓을 교정할 신호가 없다(식 13.2.2).
균등 분포의 엔트로피 $\log T=\log 16\approx 2.77$이 엔트로피의 상한이다.

In [ ]:
DIMS = [16, 64, 256] if FAST else [16, 64, 256, 1024]
init_ent = {True: [], False: []}
init_gn = {True: [], False: []}
for d in DIMS:
    for scaled in [True, False]:
        es, gs = [], []
        for s in range(3):
            E_cls, e_mark = make_task(d, s)
            m = init_model(d, s)
            X, y = make_batch(E_cls, e_mark, 256, np.random.default_rng(50 + s))
            _, _, ent, _, gn = forward_backward(m, X, y, np.sqrt(d) if scaled else 1.0)
            es.append(ent); gs.append(gn)
        init_ent[scaled].append((np.mean(es), np.std(es)))
        init_gn[scaled].append(np.mean(gs))
    print(f"d={d:5d}:  엔트로피  스케일O {init_ent[True][-1][0]:.2f} | 스케일X {init_ent[False][-1][0]:.2f}"
          f"   야코비안 질량  스케일O {init_gn[True][-1]:.2f} | 스케일X {init_gn[False][-1]:.2f}")

---
## 3. 학습 — 붕괴가 학습 실패로 이어지는가

$d$를 고정하고 스케일 유무만 바꿔 같은 조건에서 학습한다. 씨앗 4개.

In [ ]:
D_TRAIN = 128
N_SEEDS = 4
curves = {}
for scaled in [True, False]:
    curves[scaled] = [train(D_TRAIN, scaled, seed=s) for s in range(N_SEEDS)]
    accs = [c[0][-1, 1] for c in curves[scaled]]
    print(f"스케일 {'O' if scaled else 'X'} (d={D_TRAIN}): 최종 정확도 {np.round(accs, 2)}")

---
## 4. 교재 그림 — fig_13_2_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 초기 엔트로피 vs d
ax = axes[0]
for scaled, col, lb in [(True, CB[5], lab('$\\sqrt{d}$ 스케일', 'scaled')),
                        (False, CB[4], lab('스케일 없음', 'unscaled'))]:
    mu = [m_ for m_, s_ in init_ent[scaled]]
    sd = [s_ for m_, s_ in init_ent[scaled]]
    ax.errorbar(DIMS, mu, yerr=sd, fmt='o-', color=col, ms=5, capsize=3, label=lb)
ax.axhline(np.log(T_SEQ), color='k', lw=0.7, ls=':')
ax.text(DIMS[0], np.log(T_SEQ) + 0.06, lab('균등 분포의 엔트로피 $\\log T$', 'uniform'), fontsize=8)
ax.set_xscale('log')
ax.set_xlabel(lab('차원 $d$ (로그 축)', 'dimension $d$'))
ax.set_ylabel(lab('초기 어텐션 엔트로피', 'initial attn entropy'))
ax.set_title(lab('(a) 스케일이 없으면 초기화가 이미 포화다', '(a) initial entropy'), fontsize=10)
ax.legend(fontsize=8)

# (b) 야코비안 학습 가능 질량 vs d
ax = axes[1]
ax.loglog(DIMS, init_gn[True], 'o-', color=CB[5], ms=5, label=lab('$\\sqrt{d}$ 스케일', 'scaled'))
ax.loglog(DIMS, init_gn[False], 's-', color=CB[4], ms=5, label=lab('스케일 없음', 'unscaled'))
ax.set_xlabel(lab('차원 $d$ (로그 축)', 'dimension $d$'))
ax.set_ylabel(lab('야코비안 질량 $\\sum_j \\alpha_j(1-\\alpha_j)$', 'Jacobian mass'))
ax.set_title(lab('(b) 야코비안의 붕괴 — 주소 교정 신호의 소멸', '(b) Jacobian collapse'), fontsize=10)
ax.legend(fontsize=8)

# (c) 학습 곡선
ax = axes[2]
for scaled, col, lb in [(True, CB[5], lab('$\\sqrt{d}$ 스케일', 'scaled')),
                        (False, CB[4], lab('스케일 없음', 'unscaled'))]:
    for i, (ca, _) in enumerate(curves[scaled]):
        ax.plot(ca[:, 0], ca[:, 1], '-', color=col, lw=1.1, alpha=0.8,
                label=lb if i == 0 else None)
ax.axhline(1 / N_CLS, color='k', lw=0.6, ls=':')
ax.set_xlabel(lab('학습 걸음', 'step'))
ax.set_ylabel(lab('정확도', 'accuracy'))
ax.set_title(lab(f'(c) 학습 곡선 ($d={D_TRAIN}$, 씨앗 {N_SEEDS}개)', '(c) training'), fontsize=10)
ax.legend(fontsize=8, loc='center right')

# (d) 엔트로피 궤적
ax = axes[3]
for scaled, col, lb in [(True, CB[5], lab('$\\sqrt{d}$ 스케일', 'scaled')),
                        (False, CB[4], lab('스케일 없음', 'unscaled'))]:
    for i, (_, ce) in enumerate(curves[scaled]):
        ax.plot(ce[:, 0], ce[:, 1], '-', color=col, lw=1.1, alpha=0.8,
                label=lb if i == 0 else None)
ax.axhline(np.log(T_SEQ), color='k', lw=0.6, ls=':')
ax.set_xlabel(lab('학습 걸음', 'step'))
ax.set_ylabel(lab('어텐션 엔트로피', 'attn entropy'))
ax.set_title(lab('(d) 엔트로피 궤적 — 출발점이 운명을 가른다', '(d) entropy trajectory'), fontsize=10)
ax.legend(fontsize=8, loc='center right')

save_book_fig(fig, 'fig_13_2_4')
plt.show()

> ### 읽는 법
>
> (a) 스케일 없는 어텐션의 초기 엔트로피는 $d$와 함께 붕괴한다 — 로짓 표준편차가
> $\sqrt d$로 자라기 때문(명제 13.2.1). 스케일을 걸면 $d$와 무관하게 일정하다.
> (b) 그 대가가 학습 신호다. 야코비안 $\operatorname{diag}(\alpha)-\alpha\alpha^\top$의
> 대각합이 $d$와 함께 붕괴한다(식 13.2.2) — 로짓을 어느 방향으로 고쳐야 할지에 대한
> 신호 자체가 사라지는 것이다. (파라미터 기울기의 원시 노름은 $1/\sqrt d$ 인자와 뒤섞여
> 이 붕괴를 가리므로, 야코비안 질량이 정확한 진단 지표다.)
> (c)–(d) 결과: 스케일 없는 모델은 무작위로 찍은 주소를 교정하지 못해 네 씨앗 모두
> 우연 수준에 머문다. 엔트로피 궤적이 인상적이다 — 정상 모델은 높은 엔트로피를
> **유지한 채** 정확도 1.0에 도달한다. 표지에 필요한 만큼만 질량을 주면 되므로
> 분포가 뾰족해질 이유가 없는 것이다. 스케일 없는 모델은 낮은 곳에서 굳은 채 시작해
> 끝까지 움직이지 못한다. 뾰족함 자체가 병이 아니라, **시작부터 뾰족해 신호가 없는
> 것**이 병이다(§13.2.5).

---
## 5. 자기 점검

1. (a)의 스케일 없는 곡선이 $d=1024$에서 엔트로피 거의 0에 닿는 이유를 로짓 표준편차 $\sqrt{1024}=32$로 설명하라. §13.2.3의 손계산과 대조할 것.
2. (b) 대신 파라미터 기울기의 원시 노름 $\|\nabla_{W_K}L\|$을 재면 스케일 없는 쪽이 오히려 커 보인다. $1/\sqrt d$ 인자를 추적해 왜 그런지, 그리고 왜 그것이 학습 가능성의 지표가 못 되는지(Adam의 좌표별 정규화를 상기) 설명하라.
3. `scale=1.0` 대신 `scale=d`(과잉 스케일)로 두면 (c)(d)가 어떻게 되는가? §13.1.5(d)의 과대 평활과 연결하라.
4. 학습이 끝난 정상 모델의 로짓 표준편차를 재 보라. 초기의 1보다 커져 있는가? §13.2.6의 QK 정규화 동기와 연결하라.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `DIMS` | 2절 | ≤1024 | 붕괴의 사다리 |
| `D_TRAIN` | 3절 | 128 | 작으면 스케일 없어도 이따금 살아남는다 |
| `T_SEQ` | 1절 | 16 | 엔트로피 상한 $\log T$ |
| `lr` | 1절 | 2e-3 | 죽은 기울기를 큰 걸음으로 보상? |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")